In [36]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [38]:
csv_path = "/content/drive/MyDrive/ML/abalone.csv"
df =pd.read_csv(csv_path)

print(df.head())
print(df.columns)

  Sex  Length  Diameter  Height  Whole weight  Shucked weight  Viscera weight  \
0   M   0.455     0.365   0.095        0.5140          0.2245          0.1010   
1   M   0.350     0.265   0.090        0.2255          0.0995          0.0485   
2   F   0.530     0.420   0.135        0.6770          0.2565          0.1415   
3   M   0.440     0.365   0.125        0.5160          0.2155          0.1140   
4   I   0.330     0.255   0.080        0.2050          0.0895          0.0395   

   Shell weight  Rings  
0         0.150     15  
1         0.070      7  
2         0.210      9  
3         0.155     10  
4         0.055      7  
Index(['Sex', 'Length', 'Diameter', 'Height', 'Whole weight', 'Shucked weight',
       'Viscera weight', 'Shell weight', 'Rings'],
      dtype='object')


In [39]:
df['Sex'] = df['Sex'].map({'M': 0, 'F': 1, 'I': 2}) # Label encoding for nominal data

x = df.drop('Rings', axis=1)
y = df['Rings']
print(x.head())
print(x.shape)

   Sex  Length  Diameter  Height  Whole weight  Shucked weight  \
0    0   0.455     0.365   0.095        0.5140          0.2245   
1    0   0.350     0.265   0.090        0.2255          0.0995   
2    1   0.530     0.420   0.135        0.6770          0.2565   
3    0   0.440     0.365   0.125        0.5160          0.2155   
4    2   0.330     0.255   0.080        0.2050          0.0895   

   Viscera weight  Shell weight  
0          0.1010         0.150  
1          0.0485         0.070  
2          0.1415         0.210  
3          0.1140         0.155  
4          0.0395         0.055  
(4177, 8)


In [40]:
#  Train - Test Split
def train_test_split_numpy(x, y, test_size=0.2, seed=1):
    rng = np.random.default_rng(seed)
    n = x.shape[0]
    idx = np.arange(n)
    rng.shuffle(idx)

    test_n = int(round(n * test_size))
    test_idx = idx[:test_n]
    train_idx = idx[test_n:]

    return x[train_idx], x[test_idx], y[train_idx], y[test_idx]

In [41]:
x_train, x_test, y_train, y_test = train_test_split_numpy(x.values, y.values, test_size=0.2, seed=1)

# converting the target vectors (y) into matrix format (N, 1).
y_train = y_train.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

In [42]:
#Z-Score Normalization
def standardize_fit(x):
    mean = x.mean(axis=0)
    std = x.std(axis=0)
    std = np.where(std == 0, 1.0, std)
    return mean, std

def standardize_transform(x, mean, std):
    return (x - mean) / std

mean, std = standardize_fit(x_train)
x_train_s = standardize_transform(x_train, mean, std)
x_test_s = standardize_transform(x_test, mean, std)

In [43]:
#Size Adjustment for CNN Input:
#We have 8 features, we set the number of channels to 1 => (N, 8, 1)
x_train_cnn = x_train_s[:, :, np.newaxis]
x_test_cnn = x_test_s[:, :, np.newaxis]

In [44]:
# creating cnn class
class CNN1DRegressorFromScratch:
    def __init__(self, input_feats=8, filter_size=3, num_filters=2, seed=42):
        np.random.seed(seed)
        self.filter_size = filter_size
        self.num_filters = num_filters

        # 1. Converting Layer Parameters:Filter size= 3, Input channe= 1, Output channel= 2
        self.W_conv = np.random.randn(filter_size, 1, num_filters) * np.sqrt(2.0 / filter_size)
        self.b_conv = np.zeros((1, 1, num_filters))

        self.flatten_dim = (input_feats - filter_size + 1) * num_filters


        self.W_dense = np.random.randn(self.flatten_dim, 1) * np.sqrt(2.0 / self.flatten_dim)
        self.b_dense = np.zeros((1, 1))

    def relu(self, x):
        return np.maximum(0, x)

    def relu_derivative(self, x):
        return (x > 0).astype(float)

    def forward(self, X):
        N, L, _ = X.shape
        out_L = L - self.filter_size + 1

        # 1D Convolution proccess
        self.conv_in = X
        self.conv_out = np.zeros((N, out_L, self.num_filters))

        for f in range(self.num_filters):
            for i in range(out_L):
                window = X[:, i:i+self.filter_size, 0] # (N, filter_size)
                filt = self.W_conv[:, 0, f]            # (filter_size,)
                self.conv_out[:, i, f] = np.dot(window, filt)

        self.conv_out += self.b_conv
        self.act_out = self.relu(self.conv_out) #activation function

        #Flatten layer: (N, 6, 2) -> (N, 12)
        self.flatten_out = self.act_out.reshape(N, self.flatten_dim)

        # exit layer (Dense): (N, 12) @ (12, 1) + b = (N, 1)
        y_pred = np.dot(self.flatten_out, self.W_dense) + self.b_dense
        return y_pred

    def backward(self, d_out, lr):
        N = d_out.shape[0]

        # Dense layer Gradiants
        dW_dense = np.dot(self.flatten_out.T, d_out) # (12, N) @ (N, 1) -> (12, 1)
        db_dense = np.sum(d_out, axis=0, keepdims=True)

        d_flatten = np.dot(d_out, self.W_dense.T) # (N, 1) @ (1, 12) -> (N, 12)
        d_act = d_flatten.reshape(self.act_out.shape) # (N, 6, 2)

        # ReLU derivatives
        d_conv = d_act * self.relu_derivative(self.conv_out)

        # Conv layer Gradiants
        dW_conv = np.zeros_like(self.W_conv)
        db_conv = np.sum(d_conv, axis=(0, 1), keepdims=True)

        out_L = d_conv.shape[1]
        for f in range(self.num_filters):
            for i in range(out_L):
                window = self.conv_in[:, i:i+self.filter_size, 0] # (N, filter_size)
                d_pixel = d_conv[:, i, f] # (N,)
                dW_conv[:, 0, f] += np.dot(d_pixel, window)

        # update parameters (SGD/Gradient Descent)
        self.W_dense -= lr * dW_dense
        self.b_dense -= lr * db_dense
        self.W_conv -= lr * dW_conv
        self.b_conv -= lr * db_conv

In [45]:
# training part
cnn_model = CNN1DRegressorFromScratch(input_feats=8, filter_size=3, num_filters=2, seed=42)

epochs = 1500
learning_rate = 0.001

print("1D CNN model is training...")

for epoch in range(epochs):
    y_pred = cnn_model.forward(x_train_cnn)

    loss = np.mean((y_pred - y_train) ** 2)

    d_out = 2 * (y_pred - y_train) / x_train_cnn.shape[0]

    cnn_model.backward(d_out, learning_rate)

    if epoch % 150 == 0:
        print(f"Epoch {epoch:4d} | training MSE loss: {loss:.3f}")

1D CNN model is training...
Epoch    0 | training MSE loss: 122.317
Epoch  150 | training MSE loss: 7.849
Epoch  300 | training MSE loss: 7.012
Epoch  450 | training MSE loss: 6.666
Epoch  600 | training MSE loss: 6.428
Epoch  750 | training MSE loss: 6.239
Epoch  900 | training MSE loss: 6.077
Epoch 1050 | training MSE loss: 5.934
Epoch 1200 | training MSE loss: 5.808
Epoch 1350 | training MSE loss: 5.695


In [46]:
#R2 & MAE metrics
y_test_pred = cnn_model.forward(x_test_cnn)

def r2_score(y_true, y_pred):
    return 1 - (np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2))

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

cnn_r2 = r2_score(y_test, y_test_pred)
cnn_mae = mae(y_test, y_test_pred)

print(f"CNN R2 Score     : %{cnn_r2 * 100:.2f}")
print(f"CNN MAE fault   : {cnn_mae:.4f}")

CNN R2 Score     : %47.77
CNN MAE fault   : 1.6453
